##### Vector stores
Uma das maneiras mais comuns de armazenar e buscar dados não estruturados é realizando o embedding e armazenando os vetores resultantes e, em seguida, na hora da consulta, realizar o embedding da consulta e recuperar os vetores 'mais semelhantes'. Uma VectorStore faz o armazenamento dos vetores e a realização da busca de vetores para você.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings
from os import path

/home/greg/Documents/Estudos/CURSO_ASIMOV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text",
)

In [4]:
path_file = path.join('/', 'tmp', 'Explorando o Universo das IAs com Hugging Face.pdf')
pdf = PyPDFLoader(path_file)
docs = pdf.load()

len(docs)

89

In [5]:
chunks_size = 500
chunks_overlap = 50
separator = ["\n\n", "\n", " ", ".", ","]

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunks_size,
    chunk_overlap=chunks_overlap,
    separators=separator
)

##### Criando a base de dados

In [6]:
db_vector = path.join('/', 'tmp', 'db_vector')

vector_store = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model,
    persist_directory=db_vector
)

In [7]:
vector_store

##### Carregar documentos existentes

In [8]:
db_vector = path.join('/', 'tmp', 'db_vector')

vector_store = Chroma(
    embedding_function=embedding_model,
    persist_directory=db_vector
)

/tmp/ipykernel_186264/3430124453.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


#### Retrieval

In [11]:
pergunta = "Quais são os principais pontos do livro?"

resposta = vector_store.similarity_search(pergunta, k=2)
len(resposta), resposta

(2,
 [Document(metadata={'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'subject': '', 'trapped': '/False', 'creationdate': '2024-03-20T18:50:05-03:00', 'title': 'Explorando o Universo das IAs com Hugging Face', 'total_pages': 89, 'keywords': '', 'moddate': '2024-03-20T18:50:05-03:00', 'source': '/tmp/Explorando o Universo das IAs com Hugging Face.pdf', 'author': 'Asimov Academy', 'producer': 'pdfTeX-1.40.25', 'page_label': '73', 'creator': 'LaTeX via pandoc with the Eisvogel template', 'page': 73}, page_content='Explorando o Universo das IAs com Hugging Face\nHugging Face, desde que você utilize a configuração mais básica.\nFigure 27: Página inicial dos Spaces do Hugging Face\nEste link apresenta os principais features dos Spaces do Hugging Face.\nAsimov Academy 73'),
  Document(metadata={'creationdate': '2024-03-20T18:50:05-03:00', 'page': 73, 'title': 'Explorando o Universo das IAs com Hugging Face', 'author': 'Asi

In [12]:
for doc in resposta:
    print(doc.page_content)
    print('-'*100)


Explorando o Universo das IAs com Hugging Face
Hugging Face, desde que você utilize a configuração mais básica.
Figure 27: Página inicial dos Spaces do Hugging Face
Este link apresenta os principais features dos Spaces do Hugging Face.
Asimov Academy 73
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Hugging Face, desde que você utilize a configuração mais básica.
Figure 27: Página inicial dos Spaces do Hugging Face
Este link apresenta os principais features dos Spaces do Hugging Face.
Asimov Academy 73
----------------------------------------------------------------------------------------------------


In [16]:
resposta = vector_store.similarity_search_with_score(pergunta, k=5)
len(resposta), resposta

(5,
 [(Document(metadata={'source': '/tmp/Explorando o Universo das IAs com Hugging Face.pdf', 'subject': '', 'page_label': '73', 'moddate': '2024-03-20T18:50:05-03:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'keywords': '', 'author': 'Asimov Academy', 'total_pages': 89, 'producer': 'pdfTeX-1.40.25', 'creationdate': '2024-03-20T18:50:05-03:00', 'trapped': '/False', 'creator': 'LaTeX via pandoc with the Eisvogel template', 'title': 'Explorando o Universo das IAs com Hugging Face', 'page': 73}, page_content='Explorando o Universo das IAs com Hugging Face\nHugging Face, desde que você utilize a configuração mais básica.\nFigure 27: Página inicial dos Spaces do Hugging Face\nEste link apresenta os principais features dos Spaces do Hugging Face.\nAsimov Academy 73'),
   0.6453173160552979),
  (Document(metadata={'total_pages': 89, 'page_label': '73', 'creator': 'LaTeX via pandoc with the Eisvogel template', 'page':

In [17]:
for doc, score in resposta:
    print(doc.page_content)
    print(score)
    print('-'*100)

Explorando o Universo das IAs com Hugging Face
Hugging Face, desde que você utilize a configuração mais básica.
Figure 27: Página inicial dos Spaces do Hugging Face
Este link apresenta os principais features dos Spaces do Hugging Face.
Asimov Academy 73
0.6453173160552979
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Hugging Face, desde que você utilize a configuração mais básica.
Figure 27: Página inicial dos Spaces do Hugging Face
Este link apresenta os principais features dos Spaces do Hugging Face.
Asimov Academy 73
0.6453173160552979
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Figure 34: Erro no aplicativo
O problema aqui é que nunca especificamos as dependências (bibliotecas) que seu webapp precisa.
Para fazer isso, volte para a abaFiles, e clique emAdd file e em seguida Create

#### FAISS

In [19]:
vectors_store = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [20]:
resposta = vectors_store.similarity_search(pergunta, k=2)
len(resposta), resposta

(2,
 [Document(id='93f4a68c-95e5-414b-819d-eda2e25ce7b9', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX via pandoc with the Eisvogel template', 'creationdate': '2024-03-20T18:50:05-03:00', 'author': 'Asimov Academy', 'title': 'Explorando o Universo das IAs com Hugging Face', 'subject': '', 'keywords': '', 'moddate': '2024-03-20T18:50:05-03:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '/tmp/Explorando o Universo das IAs com Hugging Face.pdf', 'total_pages': 89, 'page': 73, 'page_label': '73'}, page_content='Explorando o Universo das IAs com Hugging Face\nHugging Face, desde que você utilize a configuração mais básica.\nFigure 27: Página inicial dos Spaces do Hugging Face\nEste link apresenta os principais features dos Spaces do Hugging Face.\nAsimov Academy 73'),
  Document(id='6661c3b8-d4fd-4288-b43a-e3b82acf2a49', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'La

In [21]:
for doc in resposta:
    print(doc.page_content)
    print('-'*100)

Explorando o Universo das IAs com Hugging Face
Hugging Face, desde que você utilize a configuração mais básica.
Figure 27: Página inicial dos Spaces do Hugging Face
Este link apresenta os principais features dos Spaces do Hugging Face.
Asimov Academy 73
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Figure 34: Erro no aplicativo
O problema aqui é que nunca especificamos as dependências (bibliotecas) que seu webapp precisa.
Para fazer isso, volte para a abaFiles, e clique emAdd file e em seguida Create a new file
(note também que alguns arquivos de configuração foram criados pelo Hugging Face, além do seu
app.py):
Asimov Academy 79
----------------------------------------------------------------------------------------------------


In [22]:
resposta = vectors_store.similarity_search_with_score(pergunta, k=5)
len(resposta), resposta

(5,
 [(Document(id='93f4a68c-95e5-414b-819d-eda2e25ce7b9', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX via pandoc with the Eisvogel template', 'creationdate': '2024-03-20T18:50:05-03:00', 'author': 'Asimov Academy', 'title': 'Explorando o Universo das IAs com Hugging Face', 'subject': '', 'keywords': '', 'moddate': '2024-03-20T18:50:05-03:00', 'trapped': '/False', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5', 'source': '/tmp/Explorando o Universo das IAs com Hugging Face.pdf', 'total_pages': 89, 'page': 73, 'page_label': '73'}, page_content='Explorando o Universo das IAs com Hugging Face\nHugging Face, desde que você utilize a configuração mais básica.\nFigure 27: Página inicial dos Spaces do Hugging Face\nEste link apresenta os principais features dos Spaces do Hugging Face.\nAsimov Academy 73'),
   np.float32(0.6453172)),
  (Document(id='6661c3b8-d4fd-4288-b43a-e3b82acf2a49', metadata={'producer': 'pd

In [ ]:
for doc, score in resposta:
    print(doc.page_content)
    print(score)
    print('-'*100)

Explorando o Universo das IAs com Hugging Face
Hugging Face, desde que você utilize a configuração mais básica.
Figure 27: Página inicial dos Spaces do Hugging Face
Este link apresenta os principais features dos Spaces do Hugging Face.
Asimov Academy 73
0.6453172
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
Figure 34: Erro no aplicativo
O problema aqui é que nunca especificamos as dependências (bibliotecas) que seu webapp precisa.
Para fazer isso, volte para a abaFiles, e clique emAdd file e em seguida Create a new file
(note também que alguns arquivos de configuração foram criados pelo Hugging Face, além do seu
app.py):
Asimov Academy 79
0.6461577
----------------------------------------------------------------------------------------------------
Explorando o Universo das IAs com Hugging Face
03. Testando nossa primeira IA em tempo recorde
Vamos fazer agora o nosso primeiro teste do 